# Are the Zarr's proper?

In [1]:
import numpy as np
from tifffile import xml2dict
from pathlib import Path
import glob
import napari
import zarr
import dask.array as da
import napari


In [2]:
viewer = napari.Viewer(title = 'testing drag and drop')

In [5]:
zarr_paths = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse*/zarr/*.zarr')
zarr_paths

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/zarr/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/zarr/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_11/zarr/20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/zarr/slice_5375.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/zarr/slice_5375_fast_view.ome.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_locali

In [6]:
old_zarr_fn = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/zarr/slice_5375.zarr'
new_zarr_fn =  '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/zarr/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.zarr'

In [7]:
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader

# Try reading with official OME-Zarr reader
reader = Reader(parse_url(old_zarr_fn))
nodes = list(reader())
print(f"Read {len(nodes)} nodes successfully")

Read 1 nodes successfully


In [8]:
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader

# Try reading with official OME-Zarr reader
reader = Reader(parse_url(new_zarr_fn+'/0'))
nodes = list(reader())
print(f"Read {len(nodes)} nodes successfully")

version mismatch: detected: FormatV04, requested: FormatV05


Read 1 nodes successfully


In [15]:
zarr_path = new_zarr_fn

In [22]:
[k for k in g.array_keys()]

['0', '1', '8', '4', '5', '3', '7', '6', '2']

In [24]:
# pick a scene automatically (first numeric subgroup), or set scene="0"
root = zarr.open_group(zarr_path, mode="r")
scenes = sorted([k for k in root.group_keys() if k.isdigit()], key=int)
scene = scenes[0] if scenes else ""   # if no scenes, root is the scene

# pyramid levels are ARRAYS directly under the scene -> use array_keys()
g = zarr.open_group(f"{zarr_path}/{scene}", mode="r") if scene else root
levels = sorted([k for k in g.array_keys() if k.isdigit()], key=int)

# wrap each level as a dask array
arrays = [da.from_zarr(f"{zarr_path}/{scene}/{k}" if scene else f"{zarr_path}/{k}") for k in levels]

# optional: fix big-endian (e.g. '>u2') to native
if arrays and arrays[0].dtype.byteorder == ">":
    arrays = [da.map_blocks(lambda a: a.byteswap().newbyteorder(), a, dtype=a.dtype.newbyteorder("="))
              for a in arrays]

# assume axes = (T, C, Z, Y, X) and 2× downsample per level in Y,X
scales = [(1, 1, 1, 2**i, 2**i) for i in range(len(arrays))]


In [26]:
arrays[0]

dask.array<lambda, shape=(1, 3, 11, 41702, 56218), dtype=uint16, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>

In [20]:

v = napari.Viewer()
v.add_image(arrays,channel_axis=1, scale=[(1, 2**i, 2**i) for i in range(len(arrays))])


ValueError: length of [(1, 1, 1), (1, 2, 2), (1, 4, 4), (1, 8, 8), (1, 16, 16), (1, 32, 32), (1, 64, 64), (1, 128, 128), (1, 256, 256)] must equal 3

In [23]:
import zarr, json
from pathlib import Path

def check_ngff_image(path):
    root = zarr.open_group(path, mode="r")
    scenes = sorted([k for k in root.group_keys() if k.isdigit()], key=int) or [""]
    report = {}
    for scene in scenes:
        g = root[scene] if scene else root
        ok = True; notes=[]
        ms = g.attrs.get("multiscales")
        if not ms:
            ok=False; notes.append("missing 'multiscales' on scene group")
        else:
            spec = ms[0]
            # axes
            axes = spec.get("axes")
            if not axes or not all("name" in a for a in axes):
                ok=False; notes.append("axes list missing or malformed")
            # datasets & arrays
            ds = spec.get("datasets", [])
            if not ds: 
                ok=False; notes.append("no datasets listed in multiscales")
            else:
                for i, d in enumerate(ds):
                    p = d.get("path")
                    if p is None or p not in g:
                        ok=False; notes.append(f"dataset[{i}] path '{p}' not found as array")
                    else:
                        arr = g[p]
                        if not hasattr(arr, "shape"):
                            ok=False; notes.append(f"dataset[{i}] '{p}' is not an array")
                    # scale transform
                    cts = d.get("coordinateTransformations", [])
                    if not any(t.get("type")=="scale" and "scale" in t for t in cts):
                        notes.append(f"dataset[{i}] has no scale transform (will assume identity)")
        report[scene if scene else "root"] = {"ok": ok, "notes": notes}
    return report

# usage
z = new_zarr_fn
print(check_ngff_image(z))           # likely shows root: missing, scene '0': ok
print(check_ngff_image(z + "/0"))    # should show ok: True


{'0': {'ok': True, 'notes': []}, '1': {'ok': True, 'notes': []}}
{'root': {'ok': True, 'notes': []}}


### Show differences between zarr

In [13]:
import os
import json
from pathlib import Path

def quick_zarr_check(zarr_path):
    """Quick check of Zarr structure"""
    zarr_path = Path(zarr_path)
    print(f"\n🔍 Quick check: {zarr_path.name}")
    print("-" * 40)
    
    # Check essential files
    essentials = {
        '.zgroup': 'Zarr group identifier',
        '.zattrs': 'Multiscales metadata', 
        '0/.zarray': 'Pyramid level 0 array'
    }
    
    for file, description in essentials.items():
        exists = (zarr_path / file).exists()
        status = "✅" if exists else "❌"
        print(f"{status} {file:20} {description}")
    
    # List top-level contents
    print(f"\n📁 Top-level contents:")
    for item in sorted(zarr_path.iterdir()):
        if item.is_dir():
            print(f"  📂 {item.name}/")
        else:
            print(f"  📄 {item.name}")

# Run quick check
quick_zarr_check(old_zarr_fn)
quick_zarr_check(new_zarr_fn)


🔍 Quick check: slice_5375.zarr
----------------------------------------
❌ .zgroup              Zarr group identifier
❌ .zattrs              Multiscales metadata
❌ 0/.zarray            Pyramid level 0 array

📁 Top-level contents:
  📂 0/
  📂 1/
  📂 2/
  📂 3/
  📂 4/
  📄 zarr.json

🔍 Quick check: 20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.zarr
----------------------------------------
✅ .zgroup              Zarr group identifier
✅ .zattrs              Multiscales metadata
❌ 0/.zarray            Pyramid level 0 array

📁 Top-level contents:
  📄 .zattrs
  📄 .zgroup
  📂 0/
  📂 1/
  📂 OME/


### Restructuring 

In [7]:
import zarr, json

zarr_path = new_zarr_fn
scene = "0"
g = zarr.open_group(f"{zarr_path}/{scene}", mode="a")

# detect levels
levels = sorted([k for k in g.group_keys() if k.isdigit()], key=int)

# build NGFF v0.4 multiscales
axes = [
    {"name":"t","type":"time"},
    {"name":"c","type":"channel"},
    {"name":"z","type":"space"},
    {"name":"y","type":"space"},
    {"name":"x","type":"space"},
]
datasets = [
    {"path": lvl, "coordinateTransformations":[{"type":"scale","scale":[1,1,1,2**i,2**i]}]}
    for i, lvl in enumerate(levels)
]
g.attrs["multiscales"] = [{"version":"0.4","axes":axes,"datasets":datasets}]

print("Wrote multiscales for scene", scene)


Wrote multiscales for scene 0


# Previous troubleshooting

In [1]:
import os
import glob
from pathlib import Path
from tqdm.auto import tqdm
import shutil

In [2]:
# root = Path("/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology")
root = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/")

# mouse dirs ≥ 7
mouse_dirs = sorted(
    [p for p in root.glob("mouse_*") if int(p.name.removeprefix("mouse_")) >= 7],
    key=lambda p: int(p.name.removeprefix("mouse_")),
)

# largest .ets per mouse
largest_ets_per_mouse = []
for mouse_dir in mouse_dirs:
    ets_files = list(mouse_dir.rglob("*.ets"))
    if ets_files:
        largest_ets_per_mouse.append(max(ets_files, key=lambda f: f.stat().st_size))

# map each .ets to its sibling .vsi using the 2nd parent folder name
def ets_to_vsi(ets_path: Path) -> Path:
    acq_dir = ets_path.parent.parent.name              # e.g. "_20250901_..._5555_"
    acq_base = acq_dir.strip("_")                      # -> "20250901_..._5555"
    mouse_dir = ets_path.parents[2]     # .../<mouse_N>/
    return mouse_dir / f"{acq_base}.vsi"               # .../<mouse_N>/<acq_base>.vsi

corresponding_vsi = []
for ets in largest_ets_per_mouse:
    vsi = ets_to_vsi(ets)
    corresponding_vsi.append(vsi if vsi.exists() else None)

# show pairs (and which are missing)
for ets, vsi in zip(largest_ets_per_mouse, corresponding_vsi):
    print(f"{ets}  ->  {vsi if vsi else 'NO MATCH'}")


/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/vsi/_20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557_/stack1/frame_t_0.ets  ->  /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/vsi/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.vsi
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/vsi/_20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549_/stack1/frame_t_0.ets  ->  /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/vsi/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.vsi
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/vsi/_20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550_/stack1/frame_t_0.ets  ->  /mnt/OPERA3/Nathan/data/macrohet/mtb

## This code seeminlgy worked in the terminal

This code previous worked in two stages from terminal

~/bftools/bfconvert \
-bigtiff -compression LZW \
-series 0 \
"20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.vsi" \
"exports/mtb_slice_5375_s0.ome.tif"

bfconvert \ 
mtb_slice_5375_s0.ome.tif \
mtb_slice_5375_pyr.ome.tiff \
-bigtiff \
-pyramid-scale 2 -pyramid-resolutions 5 \
-tilex 512 -tiley 512 \
-compression LZW


# Trying bioformats2raw, in the terminal seems to work best

In [5]:
test_input = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/vsi/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.vsi'
test_output = '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.zarr'

In [ ]:
export JAVA_TOOL_OPTIONS="-Xmx64g"

In [6]:
!/home/dayn/miniconda3/envs/godspee/bin/bioformats2raw \
  "/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/vsi/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.vsi" \
  "/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.zarr"


2025-10-28 15:54:41,918 [pool-1-thread-2] ERROR c.g.bioformats2raw.Converter - Failure processing chunk; resolution=0 plane=1 xx=0 yy=0 zz=0 width=1024 height=1024 depth=1
java.lang.NoClassDefFoundError: Could not initialize class org.blosc.IBloscDll
	at org.blosc.JBlosc.compressCtx(JBlosc.java:213)
	at com.bc.zarr.CompressorFactory$BloscCompressor.compress(CompressorFactory.java:346)
	at com.bc.zarr.chunk.ChunkReaderWriterImpl_Short.write(ChunkReaderWriterImpl_Short.java:83)
	at com.bc.zarr.ZarrArray.write(ZarrArray.java:239)
	at com.glencoesoftware.bioformats2raw.Converter.writeBytes(Converter.java:1790)
	at com.glencoesoftware.bioformats2raw.Converter.processChunk(Converter.java:2021)
	at com.glencoesoftware.bioformats2raw.Converter.lambda$saveResolutions$5(Converter.java:2176)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1128)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:628)
	at java.base/java.l


KeyboardInterrupt



Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3171, in _run_cell
    result = runner(coro)
             ^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/IPython/core/async_helpers.py", line 128, in _pseudo_sync_runner
    coro.send(None)
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3413, in run_cell_async
    self._format_exception_for_storage(result.error_in_exec)
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3467, in _format_exception_for_storage
    stb = self.InteractiveTB.structured_traceback(
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/IPython/core/ultratb.py", line 1179, in structured_traceback
    return FormattedTB.structured_tracebac

In [10]:
vsi_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_*/vsi/*.vsi')
vsi_fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/vsi/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.vsi',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/vsi/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.vsi',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/vsi/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.vsi',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_11/vsi/20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555.vsi']

In [12]:
zarr_fns = [i.replace('vsi', 'zarr') for i in vsi_fns]
zarr_fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/zarr/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/zarr/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/zarr/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_11/zarr/20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555.zarr']

# Trying to load a previously converted tiff and save as zarr

Are these pyramidal tifs? - NO

In [10]:
tif_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/temp_*mouse*/tif/*')
tif_fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/temp_ignore_mouse_1/tif/mtb_slice_5375_s0.ome.tif',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/temp_ignore_mouse_1/tif/mtb_slice_5375_pyr.ome.tiff']

In [16]:
tif_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse*/tif/*ome*')
tif_fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/tif/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.ome.tif',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/tif/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.ome.tif',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/tif/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.ome.tif',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_11/tif/20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555.ome.tif']

In [17]:
tif_fn = tif_fns[0]
tif_fn

'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/tif/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.ome.tif'

In [21]:
from pathlib import Path
import tifffile as tf

def summarize_tiff(fp: Path):
    with tf.TiffFile(str(fp)) as tif:
        print(f"\n🗂  {fp.name}")
        print("   BigTIFF:", tif.is_bigtiff)
        print("   series:", len(tif.series))
        for si, s in enumerate(tif.series):
            axes = getattr(s, "axes", None)
            nlevels = len(getattr(s, "levels", []))
            print(f"   - series {si}: shape={s.shape}, axes={axes}, levels={nlevels}")
            if nlevels:
                shapes = [lev.shape for lev in s.levels]
                print(f"     level shapes: {shapes}")
        # tile info from first page
        p0 = tif.pages[0]
        tile = (getattr(p0, "tilewidth", None), getattr(p0, "tilelength", None)) if p0.is_tiled else None
        print("   tiled:", p0.is_tiled, "tile=", tile)

for fn in tif_fns:
    fn = Path(fn)
    summarize_tiff(fn)


🗂  20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.ome.tif
   BigTIFF: True


<tifffile.TiffPages @16> invalid page offset 93957314450


RuntimeError: incompatible keyframe

In [23]:
summarize_tiff(Path('/mnt/DATA/mtb_tissue_localisation/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.ome.tif'))

<tifffile.TiffPages @16> invalid page offset 93957314450



🗂  20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.ome.tif
   BigTIFF: True


RuntimeError: incompatible keyframe

In [19]:
zarr_image = tifffile.imread(tif_fn, aszarr=True)

<tifffile.TiffPages @16> invalid page offset 93957314450


RuntimeError: incompatible keyframe

In [ ]:
dask_image = da.from_zarr(zarr_image)
print(output_fn, dask_image.shape)
# dask_image = dask_image.transpose(1, 0, 2, 3,)  # lazy; no data copy
# # v0.5 / Zarr v3 store
# out_zarr = tif_fn.replace('.tif', '.zarr')
# store = parse_url(out_zarr, mode="w").store
# root = zarr.group(store=store)

# # writes data + builds a 2x YX pyramid by default
# with ProgressBar():  # optional live progress
#     write_image(
#         image=dask_image,
#         group=root,
#         axes="czyx",
#         storage_options=dict(chunks=(1, 1, 512, 512)),  # (C,Z,Y,X)
#     )

# # optional: channel labels for nicer viewing in napari/viv
# add_metadata(root, {"omero": {
#     "channels": [
#         {"label": "CF405"},
#         {"label": "CF488"},
#         {"label": "CF561"},
#     ]
# }})

# # after write_image(... axes="czyx")
# level_names = sorted(root.array_keys(), key=int)   # <-- not group_keys()

# axes = [
#     {"name": "c", "type": "channel"},
#     {"name": "z", "type": "space", "unit": "micrometer"},
#     {"name": "y", "type": "space", "unit": "micrometer"},
#     {"name": "x", "type": "space", "unit": "micrometer"},
# ]

# px_z, px_y, px_x = 2.0, 0.1625, 0.1625
# datasets = []
# for i, p in enumerate(level_names):
#     datasets.append({
#         "path": p,
#         "coordinateTransformations": [
#             {"type": "scale", "scale": [1.0, px_z, px_y*(2**i), px_x*(2**i)]},  # C Z Y X
#             {"type": "translation", "translation": [0, 0, 0, 0]},
#         ]
#     })

# write_multiscales_metadata(root, datasets=datasets, axes=axes)

In [4]:
vsi_fns = [str(i) for i in corresponding_vsi]

In [5]:
vsi_fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/vsi/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.vsi',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/vsi/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.vsi',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/vsi/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.vsi',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_11/vsi/20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555.vsi']

# Previous execution

In [ ]:
output_fns = []
for input_fn in tqdm(corresponding_vsi):
    try:
        output_fn = input_fn.with_suffix('.ome.tif')
        output_fns.append(output_fn)
        !/home/dayn/miniconda3/envs/godspee/bin/bfconvert \
        -bigtiff -compression LZW \
        -tilex 512 -tiley 512 \
        -pyramid-resolutions 5 \
        -pyramid-scale 2 \
        "$input_fn" "$output_fn"
    except:
        print(input_fn, 'failed?')

### trying to use new version of bfconvert with pyramidal factored in? what about series? i think series just does one series

In [6]:
for vsi_fn in tqdm(vsi_fns):
    tif_fn = vsi_fn.replace('.vsi', 'new_version.tif')
    !bfconvert \
    -bigtiff \
    -tilex 2048 -tiley 2048 \
    "$vsi_fn" \
    "$tif_fn"

  0%|          | 0/4 [00:00<?, ?it/s]

/bin/bash: line 1: bfconvert: command not found
/bin/bash: line 1: bfconvert: command not found
/bin/bash: line 1: bfconvert: command not found
/bin/bash: line 1: bfconvert: command not found


In [7]:
!which python

/home/dayn/analysis/miniforge3/bin/python


In [11]:
!which bfconvert

### Try vsi2tif? seems just like a wrapper for bfconvert

In [5]:
!vsi2tif -i 

/bin/bash: line 1: vsi2tif: command not found


`sage: vsi2tif [-h] -i INPUT -o OUTPUT -b BFCONVERT [-c COMPRESSION] [-s TILESIZE] [-q QUALITY] [-m MAX_MEM] [-v VERBOSE] [--remove-name-spaces] [-p PLANE] [--noskip-converted] [-f EXTENSION]

vsi2tif - simple tool for converting images from cellSens VSI to Generic TIFF

options:
  -h, --help            show this help message and exit
  -i INPUT, --input INPUT
                        folder with input files
  -o OUTPUT, --output OUTPUT
                        folder for output files
  -b BFCONVERT, --bfconvert BFCONVERT
                        path to bfconvert tool
  -c COMPRESSION, --compression COMPRESSION
                        compression technique for final image - default 'jpeg'
  -s TILESIZE, --tilesize TILESIZE
                        tile size to use during both conversion steps - default 1024
  -q QUALITY, --quality QUALITY
                        compression quality used with JPEG compression - default 87
  -m MAX_MEM, --max-mem MAX_MEM
                        set maximum memory in the java vm - default 32
  -v VERBOSE, --verbose VERBOSE
                        set verbosity level - default 1
  --remove-name-spaces  replace spaces in filename with underscores in batch mode
  -p PLANE, --plane PLANE
                        image plane to convert image from. If set to -1, all series are converted and the largest is kept - default 0
  --noskip-converted    To specifically request existing files to be converted again
  -f EXTENSION, --extension EXTENSION
                        extension type to consider (e.g., .vsi)`
                    

## Convert to Zarr

In [6]:
import tifffile
import dask.array as da
import zarr
from ome_zarr.io import parse_url
from ome_zarr.writer import write_image, add_metadata
from dask.diagnostics import ProgressBar
from ome_zarr.writer import write_multiscales_metadata

In [ ]:
tif_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse*/tif/*')

In [ ]:
for tif_fn in tqdm(tif_fns):
    try:
        if 'ome' in tif_fn:
            print('skipping suspect corrupt')
            continue
        else:
            zarr_image = tifffile.imread(tif_fn, aszarr=True)
            dask_image = da.from_zarr(zarr_image)
            print(output_fn, dask_image.shape)
            dask_image = dask_image.transpose(1, 0, 2, 3,)  # lazy; no data copy
            # v0.5 / Zarr v3 store
            out_zarr = tif_fn.replace('.tif', '.zarr')
            store = parse_url(out_zarr, mode="w").store
            root = zarr.group(store=store)
            
            # writes data + builds a 2x YX pyramid by default
            with ProgressBar():  # optional live progress
                write_image(
                    image=dask_image,
                    group=root,
                    axes="czyx",
                    storage_options=dict(chunks=(1, 1, 512, 512)),  # (C,Z,Y,X)
                )
            
            # optional: channel labels for nicer viewing in napari/viv
            add_metadata(root, {"omero": {
                "channels": [
                    {"label": "CF405"},
                    {"label": "CF488"},
                    {"label": "CF561"},
                ]
            }})
        
            # after write_image(... axes="czyx")
            level_names = sorted(root.array_keys(), key=int)   # <-- not group_keys()
            
            axes = [
                {"name": "c", "type": "channel"},
                {"name": "z", "type": "space", "unit": "micrometer"},
                {"name": "y", "type": "space", "unit": "micrometer"},
                {"name": "x", "type": "space", "unit": "micrometer"},
            ]
            
            px_z, px_y, px_x = 2.0, 0.1625, 0.1625
            datasets = []
            for i, p in enumerate(level_names):
                datasets.append({
                    "path": p,
                    "coordinateTransformations": [
                        {"type": "scale", "scale": [1.0, px_z, px_y*(2**i), px_x*(2**i)]},  # C Z Y X
                        {"type": "translation", "translation": [0, 0, 0, 0]},
                    ]
                })
            
            write_multiscales_metadata(root, datasets=datasets, axes=axes)
    except:
        print(tif_fn, 'failed')

### Previous code

In [ ]:
for tif_fn in tqdm(tif_fns):
    zarr_image = tifffile.imread(tif_fn, aszarr=True)
    dask_image = da.from_zarr(zarr_image)
    print(output_fn, dask_image.shape)
    dask_image = dask_image.transpose(1, 0, 2, 3,)  # lazy; no data copy
    # v0.5 / Zarr v3 store
    out_zarr = output_fn.with_suffix('.zarr')

    store = parse_url(out_zarr, mode="w").store
    root = zarr.group(store=store)
    
    # writes data + builds a 2x YX pyramid by default
    with ProgressBar():  # optional live progress
        write_image(
            image=dask_image,
            group=root,
            axes="czyx",
            storage_options=dict(chunks=(1, 1, 512, 512)),  # (C,Z,Y,X)
        )
    
    # optional: channel labels for nicer viewing in napari/viv
    add_metadata(root, {"omero": {
        "channels": [
            {"label": "CF405"},
            {"label": "CF488"},
            {"label": "CF561"},
        ]
    }})

    # after write_image(... axes="czyx")
    level_names = sorted(root.array_keys(), key=int)   # <-- not group_keys()
    
    axes = [
        {"name": "c", "type": "channel"},
        {"name": "z", "type": "space", "unit": "micrometer"},
        {"name": "y", "type": "space", "unit": "micrometer"},
        {"name": "x", "type": "space", "unit": "micrometer"},
    ]
    
    px_z, px_y, px_x = 2.0, 0.1625, 0.1625
    datasets = []
    for i, p in enumerate(level_names):
        datasets.append({
            "path": p,
            "coordinateTransformations": [
                {"type": "scale", "scale": [1.0, px_z, px_y*(2**i), px_x*(2**i)]},  # C Z Y X
                {"type": "translation", "translation": [0, 0, 0, 0]},
            ]
        })
    
    write_multiscales_metadata(root, datasets=datasets, axes=axes)


# Arx

In [ ]:
import time
from pathlib import Path
from datetime import datetime, timedelta

# ---- config ----
ref_path = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/_20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375_/stack1/frame_t_0.ets")
out_path = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/exports/mtb_slice_5375_s0.ome.tif")

check_every_s = 90           # polling interval
min_samples_for_eta = 5      # wait this many samples before showing ETA
window_samples = 5           # moving-average window for rate
stall_threshold_gb = 0.01    # consider “no progress” if growth < this per interval
stall_checks = 60             # consecutive stalls before we say “probably done”
target_equals_ref = True     # if False, don’t compare to ref size; just use stall logic
# -----------------

def gb(bytes_): return bytes_ / (1024**3)

ref_size_gb = gb(ref_path.stat().st_size)
print(f"Reference size: {ref_size_gb:.2f} GB (for context)\n")

sizes = []
times = []
no_progress = 0

while True:
    if out_path.exists():
        s = gb(out_path.stat().st_size)
        t = time.time()
        sizes.append(s); times.append(t)

        # progress %
        pct_str = f"{(s/ref_size_gb*100):.1f}%" if ref_size_gb > 0 else "—"

        # compute smoothed rate after we have enough points
        eta_str = "estimating…"
        rate_gb_per_min = 0.0
        if len(sizes) >= min_samples_for_eta:
            w = sizes[-window_samples:]
            wt = times[-window_samples:]
            ds = w[-1] - w[0]
            dt_min = (wt[-1] - wt[0]) / 60
            if dt_min > 0 and ds > 0:
                rate_gb_per_min = ds / dt_min

                if target_equals_ref:
                    rem = max(ref_size_gb - s, 0.0)
                    if rate_gb_per_min > 0:
                        minutes = rem / rate_gb_per_min
                        eta_time = datetime.now() + timedelta(minutes=minutes)
                        eta_str = eta_time.strftime("%H:%M:%S")
                else:
                    eta_str = "—"  # unknown final size; stall detection will stop loop

        # detect stalls (don’t trigger on first sample)
        if len(sizes) >= 2:
            delta = sizes[-1] - sizes[-2]
            if delta < stall_threshold_gb:
                no_progress += 1
            else:
                no_progress = 0

        # status line
        print(
            f"\r{datetime.now().strftime('%H:%M:%S')} | "
            f"{s:7.2f} GB ({pct_str}) | "
            f"{rate_gb_per_min:5.2f} GB/min | ETA {eta_str} | "
            f"stalls {no_progress}/{stall_checks}",
            end=""
        )

        # stop conditions:
        done_by_size = target_equals_ref and (s >= 0.995 * ref_size_gb)
        done_by_stall = no_progress >= stall_checks
        if done_by_size or done_by_stall:
            print("\n✅ Done (size target reached or growth stalled).")
            break
    else:
        print("\rWaiting for output file...", end="")

    time.sleep(check_every_s)
    

Reference size: 167.93 GB (for context)

10:39:02 |   82.98 GB (49.4%) |  0.00 GB/min | ETA estimating… | stalls 0/60